# `aidp-atp` live test — API Key + inline OCI config (admin)

**Live-test row 6.** Confirms the inline-PEM OCI config path works for admin-only Database service calls (e.g. listing autonomous databases in a compartment).


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import from_inline_pem
import oci

config = from_inline_pem(
    user_ocid=os.environ['OCI_USER_OCID'],
    tenancy_ocid=os.environ['OCI_TENANCY_OCID'],
    fingerprint=os.environ['OCI_FINGERPRINT'],
    private_key_pem=os.environ['OCI_PRIVATE_KEY_PEM'],
    region=os.environ['OCI_REGION'],
)
db_client = oci.database.DatabaseClient(config=config)
adbs = db_client.list_autonomous_databases(os.environ['ATP_COMPARTMENT_OCID']).data
print('autonomous DBs visible:', len(adbs))


In [ ]:
import pandas as pd
df = spark.createDataFrame(pd.DataFrame([{'name': adb.display_name, 'state': adb.lifecycle_state} for adb in adbs]))
df.show()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-atp',
    'auth': 'apikey-admin',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
